# RoadVision AI — Détection de véhicules avec YOLOv9-C

Notebook complet : préparation des données, vérification, entraînement, évaluation et prédictions.

**Configuration demandée :** image 640, batch size 4, 10 epochs, confiance 0.25.

> Exécuter les cellules dans l'ordre. Dans Google Colab, sélectionner **Runtime → Change runtime type → T4 GPU** avant de commencer.

## 1. Vérifier l'environnement

In [ ]:
import os
import sys
import torch
import cv2
import yaml
import random
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("OpenCV :", cv2.__version__)
print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Attention : active le GPU dans les paramètres de Colab.")

## 2. Installer les dépendances et récupérer le dataset

In [ ]:
%pip install -q kagglehub pyyaml pandas matplotlib opencv-python

In [ ]:
import kagglehub
from pathlib import Path

download_path = Path(kagglehub.dataset_download(
    "ashfakyeafi/road-vehicle-images-dataset"
))
print("Dataset téléchargé dans :", download_path)

# Le dossier de données peut être directement à la racine ou dans trafic_data.
DATA_SOURCE = download_path
if not (DATA_SOURCE / "train" / "images").exists():
    train_image_dirs = [
        p for p in download_path.rglob("images")
        if p.parent.name == "train" and (p.parent / "labels").exists()
    ]
    if not train_image_dirs:
        raise FileNotFoundError("Impossible de trouver le dossier train/images.")
    DATA_SOURCE = train_image_dirs[0].parent.parent

print("Dossier dataset :", DATA_SOURCE)
print("Contenu :", [p.name for p in DATA_SOURCE.iterdir()])

## 3. Trouver et lire la configuration du dataset

In [ ]:
yaml_candidates = list(download_path.rglob("data.yaml")) + list(download_path.rglob("data.yml"))
source_yaml = yaml_candidates[0] if yaml_candidates else None

if source_yaml:
    with open(source_yaml, "r", encoding="utf-8") as f:
        source_config = yaml.safe_load(f)
    print("Configuration trouvée :", source_yaml)
    print(source_config)
else:
    source_config = {}
    print("Aucun data.yaml trouvé. Les noms des classes devront être définis manuellement.")

names = source_config.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]
if not names:
    raise ValueError("Les noms des classes sont absents du data.yaml. Définis-les dans la cellule suivante.")

CLASS_NAMES = list(names)
NUM_CLASSES = len(CLASS_NAMES)
print("Classes :", CLASS_NAMES)
print("Nombre de classes :", NUM_CLASSES)

## 4. Vérifier les images et les annotations

Format d'une ligne YOLO :

`class_id x_center y_center width height`

Les coordonnées doivent être normalisées entre 0 et 1. Les images sans fichier `.txt` correspondant sont exclues. Les annotations vides sont conservées : elles peuvent représenter une image sans objet annoté.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
source_splits = {}

for split in ["train", "valid"]:
    image_dir = DATA_SOURCE / split / "images"
    label_dir = DATA_SOURCE / split / "labels"
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f"Dossiers images/labels introuvables pour {split}")
    source_splits[split] = (image_dir, label_dir)
    print(f"{split}: {len(list(image_dir.iterdir()))} images, {len(list(label_dir.glob('*.txt')))} labels")

In [ ]:
from PIL import Image

def validate_pair(image_path, label_path):
    try:
        with Image.open(image_path) as image:
            image.verify()
    except Exception:
        return False, "image_corrompue"

    if not label_path.exists():
        return False, "annotation_manquante"

    lines = label_path.read_text(encoding="utf-8").splitlines()
    for line in lines:
        values = line.split()
        if len(values) != 5:
            return False, "format_invalide"
        try:
            class_id = int(values[0])
            x, y, w, h = map(float, values[1:])
        except ValueError:
            return False, "valeur_invalide"
        if not 0 <= class_id < NUM_CLASSES:
            return False, "classe_invalide"
        if not all(0 <= v <= 1 for v in (x, y, w, h)) or w <= 0 or h <= 0:
            return False, "coordonnees_invalides"

    return True, "valide"

valid_pairs = {"train": [], "valid": []}
issues = []

for split, (image_dir, label_dir) in source_splits.items():
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        label_path = label_dir / f"{image_path.stem}.txt"
        ok, reason = validate_pair(image_path, label_path)
        if ok:
            valid_pairs[split].append((image_path, label_path))
        else:
            issues.append((split, image_path.name, reason))

print("Paires valides :", {k: len(v) for k, v in valid_pairs.items()})
print("Fichiers à examiner :", len(issues))
if issues:
    display(pd.DataFrame(issues, columns=["split", "image", "problème"]).head(20))

## 5. Créer les ensembles train, valid et test

Le brief demande 405 images de test et 298 images de validation. Le notebook respecte ces nombres si les données disponibles le permettent. Il ne crée pas artificiellement des images : si le dataset contient moins d'images, les effectifs réels sont affichés.

In [ ]:
OUTPUT_DIR = Path("/content/roadvision_dataset")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ["train", "valid", "test"]:
    (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

train_pool = valid_pairs["train"].copy()
valid_pool = valid_pairs["valid"].copy()

random.Random(SEED).shuffle(train_pool)
random.Random(SEED).shuffle(valid_pool)

TEST_COUNT = 405
VALID_COUNT = 298

if len(train_pool) <= TEST_COUNT:
    raise ValueError(f"Il faut plus de {TEST_COUNT} images train valides. Disponibles : {len(train_pool)}")

test_pairs = train_pool[:TEST_COUNT]
train_pairs = train_pool[TEST_COUNT:]
valid_selected = valid_pool[:min(VALID_COUNT, len(valid_pool))]

def copy_pairs(pairs, split):
    for image_path, label_path in pairs:
        shutil.copy2(image_path, OUTPUT_DIR / split / "images" / image_path.name)
        shutil.copy2(label_path, OUTPUT_DIR / split / "labels" / label_path.name)

copy_pairs(train_pairs, "train")
copy_pairs(valid_selected, "valid")
copy_pairs(test_pairs, "test")

counts = {
    split: len(list((OUTPUT_DIR / split / "images").iterdir()))
    for split in ["train", "valid", "test"]
}
print("Répartition réelle :", counts)
print("Répartition demandée dans le brief : train 2297, valid 298, test 405")
if counts["train"] != 2297 or counts["valid"] != 298 or counts["test"] != 405:
    print("Remarque : les effectifs réels diffèrent du brief, car ils dépendent des fichiers valides disponibles dans le dataset téléchargé.")

## 6. Vérifier la répartition des classes

In [ ]:
class_counts = {name: 0 for name in CLASS_NAMES}

for label_file in (OUTPUT_DIR / "train" / "labels").glob("*.txt"):
    for line in label_file.read_text(encoding="utf-8").splitlines():
        if line.strip():
            class_id = int(line.split()[0])
            class_counts[CLASS_NAMES[class_id]] += 1

class_df = pd.DataFrame({
    "Classe": list(class_counts.keys()),
    "Nombre d'objets": list(class_counts.values())
})
display(class_df)

plt.figure(figsize=(9, 4))
plt.bar(class_df["Classe"], class_df["Nombre d'objets"])
plt.xticks(rotation=35, ha="right")
plt.title("Répartition des classes — train")
plt.ylabel("Nombre d'objets annotés")
plt.tight_layout()
plt.show()

## 7. Visualiser quelques images et annotations

In [ ]:
def show_yolo_sample(image_path, label_path):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]

    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            class_id, xc, yc, bw, bh = map(float, line.split())
            x1 = int((xc - bw / 2) * width)
            y1 = int((yc - bh / 2) * height)
            x2 = int((xc + bw / 2) * width)
            y2 = int((yc + bh / 2) * height)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, CLASS_NAMES[int(class_id)], (x1, max(20, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

sample_images = list((OUTPUT_DIR / "train" / "images").glob("*"))[:5]
for image_path in sample_images:
    show_yolo_sample(image_path, OUTPUT_DIR / "train" / "labels" / f"{image_path.stem}.txt")

## 8. Créer le fichier data.yaml

In [ ]:
data_config = {
    "train": str(OUTPUT_DIR / "train" / "images"),
    "val": str(OUTPUT_DIR / "valid" / "images"),
    "test": str(OUTPUT_DIR / "test" / "images"),
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES
}

DATA_YAML = OUTPUT_DIR / "data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, sort_keys=False, allow_unicode=True)

print(DATA_YAML.read_text())

## 9. Trouver le dépôt YOLOv9

In [ ]:
from pathlib import Path

possible_repos = [
    Path("/content/yolov9"),
    DATA_SOURCE / "yolov9",
    download_path / "trafic_data" / "yolov9"
]

YOLO_REPO = next((p for p in possible_repos if (p / "train_dual.py").exists()), None)

if YOLO_REPO is None:
    result = subprocess.run(
        ["git", "clone", "https://github.com/WongKinYiu/yolov9.git", "/content/yolov9"],
        check=True
    )
    YOLO_REPO = Path("/content/yolov9")

print("Dépôt YOLOv9 :", YOLO_REPO)
assert (YOLO_REPO / "train_dual.py").exists()
assert (YOLO_REPO / "val_dual.py").exists()
assert (YOLO_REPO / "detect_dual.py").exists()

In [ ]:
import os
os.chdir(YOLO_REPO)
%pip install -q -r requirements.txt

## 10. Vérifier les fichiers du modèle et les poids préentraînés

In [ ]:
MODEL_CONFIG = YOLO_REPO / "models" / "detect" / "yolov9-c.yaml"
HYP_CONFIG = YOLO_REPO / "data" / "hyps" / "hyp.scratch-high.yaml"

# Utiliser les poids déjà présents dans le dataset, sinon télécharger les poids officiels.
weight_candidates = [
    DATA_SOURCE / "yolov9-c.pt",
    download_path / "trafic_data" / "yolov9-c.pt",
    YOLO_REPO / "yolov9-c.pt"
]
WEIGHTS = next((p for p in weight_candidates if p.exists()), YOLO_REPO / "yolov9-c.pt")

if not WEIGHTS.exists():
    import urllib.request
    weight_url = "https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt"
    urllib.request.urlretrieve(weight_url, WEIGHTS)

print("Configuration :", MODEL_CONFIG.exists(), MODEL_CONFIG)
print("Hyperparamètres :", HYP_CONFIG.exists(), HYP_CONFIG)
print("Poids :", WEIGHTS.exists(), WEIGHTS)

assert MODEL_CONFIG.exists() and HYP_CONFIG.exists() and WEIGHTS.exists()

## 11. Entraîner YOLOv9-C

Le fine-tuning réutilise les poids appris sur un grand dataset, puis les ajuste sur les catégories de véhicules de RoadVision AI.

Paramètres : `img=640`, `batch=4`, `epochs=10`.

In [ ]:
# Compatibilité avec certains checkpoints PyTorch de confiance.
_original_torch_load = torch.load

def torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = torch_load_compat

if not torch.cuda.is_available():
    raise RuntimeError("GPU non disponible. Dans Colab : Runtime > Change runtime type > T4 GPU, puis relance les cellules.")

DEVICE = "0"
print("GPU utilisé :", torch.cuda.get_device_name(0))

In [ ]:
os.chdir(YOLO_REPO)

train_command = [
    sys.executable, "train_dual.py",
    "--workers", "2",
    "--device", DEVICE,
    "--batch", "4",
    "--data", str(DATA_YAML),
    "--img", "640",
    "--cfg", str(MODEL_CONFIG),
    "--weights", str(WEIGHTS),
    "--name", "roadvision_yolov9c",
    "--hyp", str(HYP_CONFIG),
    "--epochs", "10",
    "--close-mosaic", "2",
    "--exist-ok"
]

print(" ".join(train_command))
subprocess.run(train_command, check=True)

## 12. Récupérer le meilleur checkpoint

In [ ]:
RUN_DIR = YOLO_REPO / "runs" / "train" / "roadvision_yolov9c"
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"

print("best.pt :", BEST_WEIGHTS.exists())
print("last.pt :", LAST_WEIGHTS.exists())

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError("best.pt introuvable. Vérifie la sortie de l'entraînement.")

## 13. Évaluer sur le jeu de test

In [ ]:
os.chdir(YOLO_REPO)

val_command = [
    sys.executable, "val_dual.py",
    "--data", str(DATA_YAML),
    "--img", "640",
    "--batch", "4",
    "--conf-thres", "0.001",
    "--iou-thres", "0.7",
    "--device", DEVICE,
    "--weights", str(BEST_WEIGHTS),
    "--task", "test",
    "--name", "roadvision_test",
    "--exist-ok"
]

print(" ".join(val_command))
subprocess.run(val_command, check=True)

In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    results = pd.read_csv(results_csv)
    display(results.tail())
else:
    print("results.csv non trouvé dans le dossier d'entraînement.")

## 14. Générer les prédictions avec un seuil de confiance de 0.25

In [ ]:
PREDICTION_DIR = YOLO_REPO / "runs" / "detect" / "roadvision_predictions"

detect_command = [
    sys.executable, "detect_dual.py",
    "--weights", str(BEST_WEIGHTS),
    "--source", str(OUTPUT_DIR / "test" / "images"),
    "--img", "640",
    "--conf-thres", "0.25",
    "--device", DEVICE,
    "--project", str(YOLO_REPO / "runs" / "detect"),
    "--name", "roadvision_predictions",
    "--exist-ok"
]

print(" ".join(detect_command))
subprocess.run(detect_command, check=True)
print("Prédictions enregistrées dans :", PREDICTION_DIR)

In [ ]:
prediction_images = [
    p for p in PREDICTION_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
]

print("Nombre d'images prédites :", len(prediction_images))

for image_path in prediction_images[:5]:
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(9, 5))
    plt.imshow(image)
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

## 15. Analyse des résultats

In [ ]:
print("Nombre d'images train :", counts["train"])
print("Nombre d'images validation :", counts["valid"])
print("Nombre d'images test :", counts["test"])
print("Classes :", CLASS_NAMES)
print("Checkpoint évalué :", BEST_WEIGHTS)
print("Dossier des prédictions :", PREDICTION_DIR)

### Points à commenter dans le rapport

- **Precision** : proportion des détections prédites qui sont correctes.
- **Recall** : proportion des objets réels qui ont été détectés.
- **mAP@50** : moyenne de la précision moyenne avec un IoU de 0.50.
- **mAP@50–95** : moyenne des mAP pour des seuils IoU de 0.50 à 0.95.
- Comparer les résultats par classe et relier les faibles scores au nombre d'annotations, à leur qualité et à la difficulté des images.
- Les performances après 10 epochs sont les résultats mesurés. Augmenter les epochs, équilibrer les classes ou enrichir les données sont des pistes à tester, pas des résultats acquis.

## 16. Sauvegarder les livrables

Le dossier `roadvision_dataset` contient les données organisées et `runs/train/roadvision_yolov9c/weights` contient les checkpoints. Télécharge ou copie ces éléments vers ton espace de stockage avant de fermer la session Colab.

In [ ]:
print("Dataset préparé :", OUTPUT_DIR)
print("Configuration :", DATA_YAML)
print("Meilleur modèle :", BEST_WEIGHTS)
print("Dernier checkpoint :", LAST_WEIGHTS)
print("Résultats d'entraînement :", RUN_DIR / "results.csv")
print("Prédictions :", PREDICTION_DIR)